# Silver — GDP Indicators

Cleans and enriches Bronze GDP data: adds a proper date column for each quarter and rounds values.

**Source:** `bronze.statistics_iceland_gdp`  
**Output:** `silver.gdp_indicators` (Delta table)  
**Date rule:** Q1 → Jan 1, Q2 → Apr 1, Q3 → Jul 1, Q4 → Oct 1  

In [ ]:
from pyspark.sql import functions as F

df = spark.read.format("delta").table("bronze.statistics_iceland_gdp")

print(f"Bronze rows: {df.count()}")
df.show()

In [ ]:
# Derive first month of each quarter (Q1=1, Q2=4, Q3=7, Q4=10)
df_silver = (
    df.select(
        F.col("quarter"),
        F.col("year"),
        F.col("q"),
        F.to_date(
            F.concat_ws("-", F.col("year"), (F.col("q") * 3 - 2)), "yyyy-M"
        ).alias("date"),
        F.col("metric"),
        F.round("value", 2).alias("value"),
        F.current_timestamp().alias("processed_at"),
    )
    .orderBy("year", "q")
)

df_silver.show()

In [ ]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.gdp_indicators")

print(f"Saved to silver.gdp_indicators — {df_silver.count()} rows")